# Case Study 4 v2: 3D Rotating Boundary

3D bunny-ear container with the v7 curvilinear solver.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pre_post_processing_3D_v5 import (
    bunny_ear_surface,
    make_initial_conditions_3d,
    plot_initial_configuration_3d,
    set_axes_equal_3d,
    save_rotating_3d_simulation_gif,
    plot_two_snapshot_comparison_3d,
)

import importlib.util
import time
from pathlib import Path
from IPython.display import Image, display

CASE_DIR = Path.cwd()
VIDEO_DIR = CASE_DIR / "Videos"
VIDEO_DIR.mkdir(exist_ok=True)
FIGURE_DIR = CASE_DIR / "Figures"
FIGURE_DIR.mkdir(exist_ok=True)

solver_path = CASE_DIR / "curvilinear_solver_3D-rotating_v9.py"
spec = importlib.util.spec_from_file_location("csol3d_v7", solver_path)
csol3d = importlib.util.module_from_spec(spec)
spec.loader.exec_module(csol3d)

In [ ]:
# Parameters for the bunny-ear star-shaped surface
R0 = 1.0
eps = 0.55
phi0 = 0.35      # location of the ears near the north pole
sigma = 0.20     # ear width

## Boundary Shape

In [ ]:
X, Y, Z, Theta, Phi, R = bunny_ear_surface(
    R0=R0,
    eps=eps,
    phi0=phi0,
    sigma=sigma,
    n_theta=240,
    n_phi=160,
)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(X, Y, Z, linewidth=0, antialiased=True, alpha=0.95)
#ax.set_title("Bunny-ear star-shaped 3D surface")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
set_axes_equal_3d(ax, X, Y, Z)
plt.tight_layout()


fig.savefig(FIGURE_DIR / "bunny.png", dpi=300, bbox_inches="tight")

## Insert Particles

In [ ]:
N = 350
particle_radius = 0.045

q0, p0, x0, v0, m, rad, group_mobile, nearest, info = make_initial_conditions_3d(
    N=N,
    particle_radius=particle_radius,
    particle_mass=1.0,
    velocity=(0.0, 0.0, 0.0),
    fraction=1.0,
    r_min=0.06,
    r_max=0.92,
    safety=1.05,
    max_trials=500000,
    seed=4,
    R0=R0,
    eps=eps,
    phi0=phi0,
    sigma=sigma,
)

info

In [ ]:
fig, axes = plot_initial_configuration_3d(
    q0,
    x0,
    R0=R0,
    eps=eps,
    phi0=phi0,
    sigma=sigma,
    particle_radius=particle_radius,
    s=8,
)

## 3D Rotating Simulation


In [ ]:
# Parameters.
dt = 1.0e-4
T_max = 20
save_every = 200

k_contact = 1.0e4
gamma_contact = 15.0
k_w = 1.0e4
gamma_w = 15.0

g_magnitude = 9.81
omega = 2.0 * np.pi * 0.25


In [ ]:
# One-step warm-up so Numba compilation is not included in the measured run.
_ = csol3d.simulate_curvilinear_particles(
    q0=q0,
    p0=p0,
    m=m,
    rad=rad,
    dt=dt,
    T_max=dt,
    group_mobile=group_mobile,
    k_contact=k_contact,
    gamma_contact=gamma_contact,
    k_w=k_w,
    gamma_w=gamma_w,
    g_magnitude=g_magnitude,
    omega=omega,
    R0=R0,
    eps=eps,
    phi0=phi0,
    sigma=sigma,
    save_every=1,
    r_cap=0.05,
    r_exit=0.075,
    axis_cap=0.10,
    axis_exit=0.15,
)
print("Numba warm-up complete")


In [ ]:
start = time.perf_counter()
t_out, q_out, p_out, x_body_out, v_body_out, x_lab_out = csol3d.simulate_curvilinear_particles(
    q0=q0,
    p0=p0,
    m=m,
    rad=rad,
    dt=dt,
    T_max=T_max,
    group_mobile=group_mobile,
    k_contact=k_contact,
    gamma_contact=gamma_contact,
    k_w=k_w,
    gamma_w=gamma_w,
    g_magnitude=g_magnitude,
    omega=omega,
    R0=R0,
    eps=eps,
    phi0=phi0,
    sigma=sigma,
    save_every=save_every,
    r_cap=0.05,
    r_exit=0.075,
    axis_cap=0.10,
    axis_exit=0.15,
)

runtime = time.perf_counter() - start
print(f"saved {len(t_out)} frames, final t={float(t_out[-1]):.3f}, runtime={runtime:.2f} s")


In [ ]:
gif_path = VIDEO_DIR / "case_study_4_v2_3d_rotating.gif"
save_rotating_3d_simulation_gif(
    gif_path,
    t=t_out,
    q=q_out,
    x_lab=x_lab_out,
    R0=R0,
    eps=eps,
    phi0=phi0,
    sigma=sigma,
    omega=omega,
    frame_stride=1,
    fps=10,
    title_font_size=24,
    label_font_size=20,
    time_font_size=20,
    size=(1000, 600),
    particle_radius_px=2,
)

display(Image(filename=str(gif_path)))
gif_path


In [ ]:
fig, axes = plot_two_snapshot_comparison_3d(
    t_out,
    q_out,
    x_lab_out,
    time_a= 2.0,
    time_b= 9.0,
    x_body=x_body_out,
    omega=omega,
    space="both",
)
fig.savefig(FIGURE_DIR / "two_snapshots.png", dpi=300, bbox_inches="tight")